In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [2]:
df = pd.read_excel("KARCAN DATASET.xlsx")

X = df.drop(columns=["sure(dakika)", "Kod", "ucret"])
y = df["sure(dakika)"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = ["kose turu"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ]
)


In [3]:
from sklearn.linear_model import Lasso

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lasso = Pipeline(steps=[
    ("prep", preprocess),
    ("model", Lasso(alpha=0.05))
])

lasso.fit(X_train, y_train)

feature_names = (
    num_cols.tolist() +
    list(lasso.named_steps["prep"]
         .named_transformers_["cat"]
         .get_feature_names_out(cat_cols))
)

coef = lasso.named_steps["model"].coef_

importance = pd.Series(coef, index=feature_names)\
               .sort_values(key=abs, ascending=False)

print(importance)


saft cap                 3.868160
l2                       3.415462
Z                        2.268605
kose turu_KESKİN KÖŞE   -1.411691
l3                      -0.540407
clearence                0.517327
kose turu_RADIUS         0.236636
on cap                   0.000000
ara bosaltma cap         0.000000
kose degeri              0.000000
dtype: float64


In [4]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42))
])

rf.fit(X_train, y_train)

importances = rf.named_steps["model"].feature_importances_

rf_imp = pd.Series(importances, index=feature_names)\
           .sort_values(ascending=False)

print(rf_imp)


l2                       0.683710
on cap                   0.132123
saft cap                 0.068146
Z                        0.049651
ara bosaltma cap         0.024290
kose degeri              0.013784
clearence                0.011579
l3                       0.007617
kose turu_RADIUS         0.006243
kose turu_KESKİN KÖŞE    0.002857
dtype: float64


In [5]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = rf.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 :", r2_score(y_test, y_pred))


MAE: 1.7375977011494237
R2 : 0.8827743158397439


In [6]:
from sklearn.linear_model import Ridge

ridge = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)

pd.Series(
    ridge.named_steps["model"].coef_,
    index=feature_names
).sort_values(key=abs, ascending=False)


l2                       3.488185
saft cap                 2.933350
Z                        2.351216
kose turu_KESKİN KÖŞE   -1.661238
l3                      -1.290437
ara bosaltma cap         0.809644
on cap                   0.804909
clearence                0.583499
kose turu_RADIUS         0.422339
kose degeri             -0.016124
dtype: float64

In [10]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42))
])

rf.fit(X_train, y_train)

importances = rf.named_steps["model"].feature_importances_

rf_imp = pd.Series(importances, index=feature_names)\
           .sort_values(ascending=False)

print(rf_imp)


l2                       0.683710
on cap                   0.132123
saft cap                 0.068146
Z                        0.049651
ara bosaltma cap         0.024290
kose degeri              0.013784
clearence                0.011579
l3                       0.007617
kose turu_RADIUS         0.006243
kose turu_KESKİN KÖŞE    0.002857
dtype: float64


In [11]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = Pipeline([
    ("prep", preprocess),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42))
])

gbr.fit(X_train, y_train)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['on cap', 'saft cap', 'ara bosaltma cap', 'l2', 'l3', 'clearence',
       'kose degeri', 'Z'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['kose turu'])])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, n_estimators=300,
                                           random_state=42))])

In [12]:
from sklearn.model_selection import cross_val_score

models = {
    "RF": rf,
    "GBR": gbr,
    "Ridge": ridge
}

for name, model in models.items():
    score = cross_val_score(
        model, X, y,
        cv=5,
        scoring="neg_mean_absolute_error"
    )
    print(name, -score.mean())


RF 3.2671091954022997
GBR 3.153226641280798
Ridge 2.8617839361022233


In [13]:
final_model = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0))
])

final_model.fit(X, y)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['on cap', 'saft cap', 'ara bosaltma cap', 'l2', 'l3', 'clearence',
       'kose degeri', 'Z'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['kose turu'])])),
                ('model', Ridge())])

In [15]:
import pandas as pd

def sure_tahmin(input_dict):
    df_input = pd.DataFrame([input_dict])
    return final_model.predict(df_input)[0]


In [30]:
sure_tahmin({
    "on cap": 20,
    "saft cap": 20,
    "ara bosaltma cap": 0,
    "l2": 38,
    "l3": 0,
    "clearence": 0,
    "kose turu": "CHAMFER",
    "kose degeri": 3,
    "Z": 4
})


np.float64(23.574415955229615)

In [28]:
import pandas as pd

def sure_tahmin_formatli(input_dict):
    dakika = final_model.predict(pd.DataFrame([input_dict]))[0]
    dk = int(dakika)
    sn = int((dakika - dk) * 60)
    return f"{dk} dk {sn} sn"


In [34]:
sure_tahmin_formatli({
    "on cap": 10,
    "saft cap": 8,
    "ara bosaltma cap": 6,
    "l2": 15,
    "l3": 20,
    "clearence": 0.2,
    "kose turu": "RADIUS",
    "kose degeri": 0.9,
    "Z": 2
})


'2 dk 43 sn'

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# --------------------
# Veri
# --------------------
df = pd.read_excel("KARCAN DATASET.xlsx")

X = df.drop(columns=["sure(dakika)", "Kod", "ucret"])
y = df["sure(dakika)"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = ["kose turu"]

# --------------------
# Preprocess
# --------------------
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols)
])

# --------------------
# FINAL MODEL
# --------------------
final_model = Pipeline([
    ("prep", preprocess),
    ("model", Ridge(alpha=1.0))
])

final_model.fit(X, y)

# --------------------
# KAYDET
# --------------------
with open("karcanai_ridge_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("✅ Model pickle ile kaydedildi: karcanai_ridge_model.pkl")

✅ Model pickle ile kaydedildi: karcanai_ridge_model.pkl
